<a href="https://colab.research.google.com/github/chioreanufilip/lightDeepfakesSystem/blob/colab/DeepfakeDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialisation


In [ ]:
import os
import shutil
from google.colab import drive

mountpoint = '/content/drive'

# Ensure the mountpoint directory is empty if it exists and is not a mount itself.
# This addresses the error 'Mountpoint must not already contain files'.
if os.path.exists(mountpoint) and not os.path.ismount(mountpoint):
    if os.path.isdir(mountpoint) and os.listdir(mountpoint):
        print(f"Mountpoint {mountpoint} contains existing files. Clearing contents.")
        for item in os.listdir(mountpoint):
            item_path = os.path.join(mountpoint, item)
            if os.path.isfile(item_path):
                os.remove(item_path)
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
elif not os.path.exists(mountpoint):
    os.makedirs(mountpoint)

drive.mount(mountpoint, force_remount=True)

Mounted at /content/drive


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import torch.nn as nn
import torch.optim as optim
import os
save_path = '/content/drive/MyDrive/DeepfakeDetection'
if not os.path.exists(save_path):
    os.makedirs(save_path)
# Checken, ob die GPU aktiv ist
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Du arbeitest mit: {device}")

Du arbeitest mit: cuda


In [ ]:
!pip install facenet-pytorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.

In [ ]:
import cv2
from facenet_pytorch import MTCNN
from PIL import Image
import os

# Initialisiere den Face-Detector (MTCNN) auf der GPU
mtcnn = MTCNN(keep_all=False, device=device)

def extract_faces_from_video(video_path, output_folder, interval=15):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_count = 0
    video_name = os.path.basename(video_path).split('.')[0]

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        if frame_count % interval == 0:
            # Konvertiere Frame in RGB (wichtig für die KI!)
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(img)

            # Speicherpfad
            save_name = os.path.join(output_folder, f"{video_name}_f{frame_count}.jpg")

            # Gesicht finden und speichern
            # Das Modell schneidet das Gesicht automatisch aus
            try:
                mtcnn(img_pil, save_path=save_name)
                saved_count += 1
            except Exception as e:
                # Falls mal kein Gesicht im Frame gefunden wird
                pass

        frame_count += 1
    cap.release()
    return saved_count

In [ ]:
import shutil
import os

zip_quelle = '/content/drive/MyDrive/DeepfakeDetection/kaggle_faces_backup.zip'
ziel_lokal = '/content/kaggle_faces'

if os.path.exists(zip_quelle):
    print("Lade Gesichter vom Drive...")
    shutil.unpack_archive(zip_quelle, ziel_lokal, 'zip')
    print("✅ Alle Gesichter sind wieder da! Du kannst sofort mit dem Training starten.")
else:
    print("❌ Backup-Datei nicht gefunden. Prüfe deinen Google Drive Pfad!")

Lade Gesichter vom Drive...
✅ Alle Gesichter sind wieder da! Du kannst sofort mit dem Training starten.


# Data Loading test

In [ ]:
!unzip -q /content/drive/MyDrive/DeepfakeDetection/faces_backup.zip -d /

In [ ]:
import os

# Pfad zu deinen 6 Videos im Drive
video_folder = '/content/drive/MyDrive/DeepfakeDetection/'
# Pfad wo die Gesichter hin sollen
output_base = '/content/drive/MyDrive/faces'

# Erstelle Test-Labels händisch für deine 6 Dateien
# Ersetze 'dein_video_1.mp4' mit den echten Namen aus deinem Drive!
mein_test_mapping = {
    'video1.mp4': 'FAKE',
    'video2.mp4': 'FAKE',
    'video3.mp4': 'REAL',
    'video4.mp4': 'REAL',
    'video5.mp4': 'REAL',
    'video6.mp4': 'FAKE'
    # ... füge hier alle 6 hinzu
}

os.makedirs(os.path.join(output_base, 'REAL'), exist_ok=True)
os.makedirs(os.path.join(output_base, 'FAKE'), exist_ok=True)

for video_name, label in mein_test_mapping.items():
    video_path = os.path.join(video_folder, video_name)
    if os.path.exists(video_path):
        save_to = os.path.join(output_base, label)
        # Hier rufen wir deine Funktion von vorhin auf
        extract_faces_from_video(video_path, save_to, interval=10)
        print(f"Abgeschlossen: {video_name} als {label}")
    else:
        print(f"Datei nicht gefunden: {video_path}")

Abgeschlossen: video1.mp4 als FAKE
Abgeschlossen: video2.mp4 als FAKE
Abgeschlossen: video3.mp4 als REAL
Abgeschlossen: video4.mp4 als REAL
Abgeschlossen: video5.mp4 als REAL
Abgeschlossen: video6.mp4 als FAKE


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Vorbereitung der Bilder (Transformationen)
def transform_image(image_path):

  data_transforms = transforms.Compose([
      transforms.Resize((224, 224)), # MobileNetV3 Größe
      transforms.ToTensor(),         # Umwandlung in Zahlen-Matrix
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard-Werte
  ])

  # 2. Den Ordner als Datensatz laden
  # Changed path from '/content/drive/MyDrive/faces' to '/content/faces'
  # as the backup was unzipped there.
  dataset = datasets.ImageFolder(image_path, transform=data_transforms)

  # 3. Das "Fließband" erstellen (Batch Size 32 ist gut für Colab)
  train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

  print(f"Datensatz bereit! Klassen: {dataset.classes}")
  print(f"Insgesamt {len(dataset)} Bilder zum Trainieren gefunden.")
transform_image('/content/faces')

Datensatz bereit! Klassen: ['FAKE', 'REAL']
Insgesamt 180 Bilder zum Trainieren gefunden.


In [ ]:
# 1. Die Gesichter in eine ZIP-Datei packen (geht schnell)
!zip -r /content/faces_backup.zip /content/faces

# 2. Die ZIP-Datei auf dein Google Drive kopieren
import shutil
shutil.copy("/content/faces_backup.zip", "/content/drive/MyDrive/DeepfakeDetection/faces_backup.zip")

print("Backup erfolgreich! Deine Bilder liegen jetzt sicher in deinem Google Drive.")

  adding: content/faces/ (stored 0%)
  adding: content/faces/FAKE/ (stored 0%)
  adding: content/faces/FAKE/video1_f130.jpg (deflated 5%)
  adding: content/faces/FAKE/video2_f230.jpg (deflated 4%)
  adding: content/faces/FAKE/video1_f10.jpg (deflated 5%)
  adding: content/faces/FAKE/video6_f210.jpg (deflated 5%)
  adding: content/faces/FAKE/video2_f10.jpg (deflated 5%)
  adding: content/faces/FAKE/video6_f110.jpg (deflated 5%)
  adding: content/faces/FAKE/video2_f90.jpg (deflated 4%)
  adding: content/faces/FAKE/video2_f140.jpg (deflated 4%)
  adding: content/faces/FAKE/video6_f140.jpg (deflated 5%)
  adding: content/faces/FAKE/video2_f240.jpg (deflated 4%)
  adding: content/faces/FAKE/video2_f120.jpg (deflated 5%)
  adding: content/faces/FAKE/video1_f240.jpg (deflated 5%)
  adding: content/faces/FAKE/video1_f220.jpg (deflated 4%)
  adding: content/faces/FAKE/video2_f60.jpg (deflated 4%)
  adding: content/faces/FAKE/video2_f250.jpg (deflated 5%)
  adding: content/faces/FAKE/video2_f190

# Modell-Definition

In [ ]:
import torch.nn as nn
from torchvision import models

# 1. Das vortrainierte MobileNetV3 small laden
# Wir nutzen "vortrainiert", weil das Modell schon gelernt hat, Formen und Kanten zu sehen.
model = models.mobilenet_v3_small(weights='DEFAULT')

# 2. Den "Kopf" des Modells austauschen
# Das originale Modell ist für 1000 verschiedene Objekte trainiert.
# Wir brauchen aber nur 2: REAL und FAKE.
num_features = model.classifier[3].in_features

# Wir bauen einen neuen Classifier
model.classifier[3] = nn.Sequential(
    nn.Linear(num_features, 2) # 2 Ausgänge: Real oder Fake
)

# 3. Das Modell auf die GPU schieben
model = model.to(device)

print("MobileNetV3 wurde erfolgreich geladen und für Deepfake-Erkennung angepasst.")

MobileNetV3 wurde erfolgreich geladen und für Deepfake-Erkennung angepasst.


In [ ]:
import torch.optim as optim

# Die Loss-Function misst, wie falsch die KI liegt (CrossEntropy ist Standard für Klassifizierung)
criterion = nn.CrossEntropyLoss()

# Der Optimizer ist der Mechanismus, der die Gewichte im Gehirn der KI anpasst
# Wir nutzen 'Adam', einen sehr effizienten und beliebten Optimizer.
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Lehrer (Loss-Function) und Lern-Mechanismus (Optimizer) sind bereit.")

Lehrer (Loss-Function) und Lern-Mechanismus (Optimizer) sind bereit.


In [ ]:
# Den Pfad zu deinem gespeicherten Modell angeben
speicher_pfad = '/content/drive/MyDrive/DeepfakeDetection/sanity_check_modell.pth'

# Das gespeicherte Wissen (die Gewichte) in das leere Modell laden
model.load_state_dict(torch.load(speicher_pfad))
model = model.to(device)
model.eval() # Direkt in den Test-Modus schalten

print("Erfolgreich geladen! Die KI hat ihr Wissen aus der letzten Sitzung wieder.")

Erfolgreich geladen! Die KI hat ihr Wissen aus der letzten Sitzung wieder.


# Training

In [ ]:
# Anzahl der Durchläufe (Epochen)


print("Starte den Sanity Check (Test-Training)...")
def training(num_epochs):
  best_acc = 0.0
  for epoch in range(num_epochs):
      # 1. Das Modell in den Trainings-Modus versetzen
      model.train()
      running_loss = 0.0

      # Hier holt sich die KI immer 32 Bilder auf einmal vom DataLoader
      for inputs, labels in train_loader:

          # Daten auf die schnelle Grafikkarte schieben
          inputs, labels = inputs.to(device), labels.to(device)

          # WICHTIG: Das Gedächtnis des Optimizers zurücksetzen, sonst vermischt er alte mit neuen Fehlern
          optimizer.zero_grad()

          # Die KI rät: "Real oder Fake?" (Forward pass)
          outputs = model(inputs)

          # Der Lehrer berechnet, wie falsch die KI lag
          loss = criterion(outputs, labels)

          # Die KI lernt aus ihren Fehlern und passt ihre Gewichte an (Backward pass)
          loss.backward()
          optimizer.step()

          running_loss += loss.item()

      model.eval() # Modell auf "Prüfungsmodus" schalten
      val_loss = 0.0
      correct = 0
      total = 0

      with torch.no_grad(): # Wichtig: Hier wird nicht gelernt!
          for inputs, labels in val_loader:
              inputs, labels = inputs.to(device), labels.to(device)
              outputs = model(inputs)
              loss = criterion(outputs, labels)
              val_loss += loss.item()

              # Accuracy berechnen
              _, predicted = torch.max(outputs, 1)
              total += labels.size(0)
              correct += (predicted == labels).sum().item()

      # Ergebnisse anzeigen
      train_l = running_loss / len(train_loader)
      v_loss = val_loss / len(val_loader)
      v_acc = 100 * correct / total

      print(f"Epoche {epoch+1}/{num_epochs}:")
      print(f"  Train Loss: {train_l:.4f}")
      print(f"  Val Loss: {v_loss:.4f} | Val Accuracy: {v_acc:.2f}%")
      print("-" * 30)
    # In deiner training-Schleife nach der Validierung:
      if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"--- Neues bestes Modell gespeichert! (Accuracy: {best_acc:.2f}%) ---")


Starte den Sanity Check (Test-Training)...


In [ ]:
training(10)
print("Sanity Check abgeschlossen! Dein Code funktioniert.")

# Das trainierte Modell sicher auf dem Drive speichern
speicher_pfad = '/content/drive/MyDrive/DeepfakeDetection/sanity_check_modell.pth'
torch.save(model.state_dict(), speicher_pfad)
print(f"Modell erfolgreich gespeichert unter: {speicher_pfad}")

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image

# 1. Wähle dein Test-Bild aus (Füge hier den kopierten Pfad ein!)
# BEISPIEL: test_bild_pfad = '/content/drive/MyDrive/faces/FAKE/video1_f0.jpg'
test_bild_pfad = '/content/faces/FAKE/video2_f200.jpg'

# 2. Das Bild laden und vorbereiten (Die "Brille" aufsetzen)
bild = Image.open(test_bild_pfad).convert('RGB')
bild_tensor = data_transforms(bild) # Gleiche Größe & Farben wie beim Training

# WICHTIG: Die KI erwartet immer einen "Stapel" von Bildern.
# Da wir nur ein Bild haben, tricksen wir und machen einen Stapel mit 1 Bild daraus.
bild_tensor = bild_tensor.unsqueeze(0).to(device)

# 3. Den Test-Modus aktivieren
model.eval() # Sagt der KI: "Nicht mehr lernen, nur noch prüfen!"

# 4. Die Vorhersage machen
print("KI analysiert das Bild...")
with torch.no_grad(): # Wir schalten die Fehlerberechnung ab, das spart Speicher
    output = model(bild_tensor)

    # Die Ausgabe in Prozentzahlen (0-100%) umwandeln
    wahrscheinlichkeiten = F.softmax(output, dim=1)

    # Die Klasse mit der höchsten Prozentzahl gewinnt
    gewinner_index = torch.argmax(wahrscheinlichkeiten, dim=1).item()

# 5. Das Ergebnis ausgeben
klassen_namen = dataset.classes # Ist meistens ['FAKE', 'REAL']
ergebnis = klassen_namen[gewinner_index]
sicherheit = wahrscheinlichkeiten[0][gewinner_index].item() * 100

print("-" * 30)
print(f"Das Ergebnis ist da!")
print(f"Die KI sagt, dieses Bild ist: {ergebnis}")
print(f"Sie ist sich zu {sicherheit:.2f}% sicher.")
print("-" * 30)

KI analysiert das Bild...
------------------------------
Das Ergebnis ist da!
Die KI sagt, dieses Bild ist: FAKE
Sie ist sich zu 100.00% sicher.
------------------------------


# Real Data Loading

In [ ]:
import pandas as pd
import json

# Pfad zur JSON-Datei (die beim Entpacken in /content/Kaggle_Videos/ gelandet ist)
json_pfad = '/content/Kaggle_Videos/train_sample_videos/metadata.json'

with open(json_pfad) as f:
    data = json.load(f)

# Erstellt die Tabelle 'df'
df = pd.DataFrame.from_dict(data, orient='index').reset_index()
df = df.rename(columns={'index': 'video_name'})

print(f"Tabelle erstellt! Wir haben Informationen zu {len(df)} Videos.")

Tabelle erstellt! Wir haben Informationen zu 400 Videos.


In [ ]:
import os

zip_pfad = '/content/drive/MyDrive/DeepfakeDetection/deepfake-detection-challenge.zip'
ziel_ordner = '/content/Kaggle_Videos/'
# !chmod 600 /content/drive/MyDrive/DeepfakeDetection/kaggle.json
print("Entpacke die 400 Videos direkt aus deinem Google Drive...")
print("Das dauert einen Moment...")

# Erstellt den Zielordner, falls er nicht existiert
os.makedirs(ziel_ordner, exist_ok=True)

# 3. Die ZIP-Datei entpacken (auf der schnellen lokalen Colab-Festplatte)
print("Entpacke Videos...")
!unzip -q {zip_pfad} -d {ziel_ordner}

print("Download und Entpacken erfolgreich! Die echten Daten sind bereit.")

Entpacke die 400 Videos direkt aus deinem Google Drive...
Das dauert einen Moment...
Entpacke Videos...
Download und Entpacken erfolgreich! Die echten Daten sind bereit.


In [ ]:
import os
import cv2
import torch
from PIL import Image
from facenet_pytorch import MTCNN
from tqdm import tqdm
import shutil

# --- 1. VORBEREITUNG ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ziel_ordner = '/content/kaggle_faces'
os.makedirs(os.path.join(ziel_ordner, 'REAL'), exist_ok=True)
os.makedirs(os.path.join(ziel_ordner, 'FAKE'), exist_ok=True)

mtcnn = MTCNN(keep_all=False, device=device)
# Nutze den Pfad, den der "Suchhund" vorhin gefunden hat
video_ordner = '/content/Kaggle_Videos/train_sample_videos/'

# --- NEU: CHECKPOINT-SCAN ---
# Wir schauen nach, welche Videos wir schon fertig haben
print("Scanne bereits extrahierte Gesichter...")
existierende_real = {f.split('_f')[0] for f in os.listdir(os.path.join(ziel_ordner, 'REAL'))}
existierende_fake = {f.split('_f')[0] for f in os.listdir(os.path.join(ziel_ordner, 'FAKE'))}
fertige_videos = existierende_real.union(existierende_fake)
print(f"Gefunden: {len(fertige_videos)} bereits verarbeitete Videos. Diese werden übersprungen.")

# --- 2. DAS WERKZEUG ---
def extract_faces_from_video(video_path, output_folder, interval=30):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    video_name = os.path.basename(video_path).split('.')[0]

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        if frame_count % interval == 0:
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(img)
            save_name = os.path.join(output_folder, f"{video_name}_f{frame_count}.jpg")

            try:
                mtcnn(img_pil, save_path=save_name)
            except:
                pass
        frame_count += 1
    cap.release()

# --- 3. DER ARBEITER MIT ÜBERSPRUNG-LOGIK ---
print("\nStarte die Gesichtsextraktion...")

for index, row in tqdm(df.iterrows(), total=len(df)):
    video_name = row['video_name']
    video_id = video_name.split('.')[0] # Name ohne .mp4
    label = row['label']

    # --- PRÜFUNG: SCHON FERTIG? ---
    if video_id in fertige_videos:
        continue # Springt sofort zum nächsten Video in der Schleife

    video_pfad = os.path.join(video_ordner, video_name)
    speicher_pfad = os.path.join(ziel_ordner, label)

    if os.path.exists(video_pfad):
        extract_faces_from_video(video_pfad, speicher_pfad, interval=30)

print("✅ Alle 400 Videos (neu + alte) sind nun im Kasten!")

# --- 4. BACKUP (ZIP) ---
# WICHTIG: Erstelle das ZIP erst, wenn wirklich alles fertig ist
backup_pfad = '/content/drive/MyDrive/DeepfakeDetection/kaggle_faces_backup'
print("Erstelle ZIP-Archiv für Google Drive...")
shutil.make_archive(backup_pfad, 'zip', ziel_ordner)
print(f"✅ Backup erfolgreich auf Drive gesichert!")

Scanne bereits extrahierte Gesichter...
Gefunden: 397 bereits verarbeitete Videos. Diese werden übersprungen.

Starte die Gesichtsextraktion...


100%|██████████| 400/400 [00:10<00:00, 38.95it/s]


✅ Alle 400 Videos (neu + alte) sind nun im Kasten!
Erstelle ZIP-Archiv für Google Drive...
✅ Backup erfolgreich auf Drive gesichert!


# Training real Model


In [ ]:
# Installiert das kleine Helfer-Tool
!pip install split-folders tqdm

import splitfolders
import os

input_ordner = '/content/kaggle_faces'
output_ordner = '/content/dataset_split'

print("Starte die wissenschaftliche Aufteilung der Daten...")
print("Teile auf in: 70% Train, 15% Validation, 15% Test")

# Der Befehl, der die Bilder kopiert und aufteilt
# seed=42 sorgt dafür, dass die "Zufallsmischung" immer exakt gleich bleibt
splitfolders.ratio(input_ordner, output=output_ordner,
    seed=42, ratio=(.7, .15, .15), group_prefix=None, move=False)

print("-" * 30)
print("✅ FERTIG! Der Ordner 'dataset_split' wurde erstellt.")

# Wir schauen kurz rein, ob alles geklappt hat:
print("Ordner-Struktur im neuen Verzeichnis:")
print(os.listdir(output_ordner))

Starte die wissenschaftliche Aufteilung der Daten...
Teile auf in: 70% Train, 15% Validation, 15% Test


Copying files: 3862 files [00:00, 5022.39 files/s]

------------------------------
✅ FERTIG! Der Ordner 'dataset_split' wurde erstellt.
Ordner-Struktur im neuen Verzeichnis:
['train', 'val', 'test']


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Nutze Rechenpower: {device}")

# --- TEIL 1: DIE DATEN (mit Augmentation und Split) ---
print("1. Richte Daten-Kellner (DataLoaders) ein...")

# Die Brille für das Training (Mit Zufalls-Spiegeln gegen Auswendiglernen!)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),      # Bild leicht drehen
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5), # Farben extrem ändern
    transforms.RandomGrayscale(p=0.2),  # Manchmal Schwarz-Weiß (zwingt zur Formerkennung)
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Die Brille für die Prüfung (Kein Schummeln/Spiegeln erlaubt!)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Lade die drei Ordner, die wir vorhin aufgeteilt haben
ordner_basis = '/content/dataset_split/'
train_data = datasets.ImageFolder(f'{ordner_basis}/train', transform=train_transforms)
val_data = datasets.ImageFolder(f'{ordner_basis}/val', transform=val_transforms)

# Die Kellner erstellen
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

# --- TEIL 2: DAS GEHIRN & DIE BESTRAFUNG ---
print("2. Lade das MobileNetV3 und stelle die Bestrafung (Class Weights) ein...")

# MobileNet laden und den Ausgang auf 2 Klassen (Fake/Real) umbauen
model = models.mobilenet_v3_small(weights='DEFAULT')
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
model = model.to(device)

# HIER IST DEINE BESTRAFUNG! (Gewichte für die Fehlerfunktion)
# Klasse 0 = FAKE (Gewicht 1.0)
# Klasse 1 = REAL (Gewicht 4.08, weil es viermal seltener ist)
gewichte = torch.tensor([1.0, 4.08]).to(device)
criterion = nn.CrossEntropyLoss(weight=gewichte)

# Der Optimierer (Wie große Schritte das Modell beim Lernen macht)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✅ Alles bereit! Du kannst jetzt deine Trainings-Schleife starten.")

Nutze Rechenpower: cuda
1. Richte Daten-Kellner (DataLoaders) ein...
2. Lade das MobileNetV3 und stelle die Bestrafung (Class Weights) ein...
✅ Alles bereit! Du kannst jetzt deine Trainings-Schleife starten.


In [ ]:
training(10)
speicher_pfad = '/content/drive/MyDrive/DeepfakeDetection/sanity_check_modell.pth'
torch.save(model.state_dict(), speicher_pfad)
print(f"Modell erfolgreich gespeichert unter: {speicher_pfad}")

Epoche 1/10:
  Train Loss: 0.5940
  Val Loss: 0.4108 | Val Accuracy: 84.46%
------------------------------
--- Neues bestes Modell gespeichert! (Accuracy: 84.46%) ---
Epoche 2/10:
  Train Loss: 0.3926
  Val Loss: 0.3063 | Val Accuracy: 87.22%
------------------------------
--- Neues bestes Modell gespeichert! (Accuracy: 87.22%) ---
Epoche 3/10:
  Train Loss: 0.3151
  Val Loss: 0.4104 | Val Accuracy: 82.38%
------------------------------
Epoche 4/10:
  Train Loss: 0.2494
  Val Loss: 0.2311 | Val Accuracy: 91.88%
------------------------------
--- Neues bestes Modell gespeichert! (Accuracy: 91.88%) ---
Epoche 5/10:
  Train Loss: 0.1966
  Val Loss: 0.5709 | Val Accuracy: 86.70%
------------------------------
Epoche 6/10:
  Train Loss: 0.1890
  Val Loss: 0.2243 | Val Accuracy: 92.40%
------------------------------
--- Neues bestes Modell gespeichert! (Accuracy: 92.40%) ---
Epoche 7/10:
  Train Loss: 0.1733
  Val Loss: 0.2400 | Val Accuracy: 90.16%
------------------------------
Epoche 8/10

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print("Vorbereitung auf die Abschlussprüfung...")

# 1. Die Prüfungs-Brille (Kein Schummeln, keine Augmentation!)
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Den geheimen Test-Ordner laden
test_ordner = '/content/dataset_split/test'
test_data = datasets.ImageFolder(test_ordner, transform=test_transforms)

# 3. Den Test-Kellner erstellen
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Es werden {len(test_data)} unbekannte Bilder getestet.\n")
print("Prüfung läuft... Bitte warten...")

# 4. Die Abschlussprüfung
model.eval() # GANZ WICHTIG: Modell in den Prüfungsmodus schalten
test_loss = 0.0
richtige_treffer = 0
gesamt_bilder = 0

with torch.no_grad(): # Kein Lernen mehr erlaubt!
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # KI gibt ihre Antwort
        outputs = model(inputs)

        # Fehler berechnen
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        # Zählen, ob die Antwort richtig war
        _, vorhersage = torch.max(outputs, 1)
        gesamt_bilder += labels.size(0)
        richtige_treffer += (vorhersage == labels).sum().item()

# 5. Finale Noten berechnen
test_loss_durchschnitt = test_loss / len(test_loader)
test_genauigkeit = 100 * richtige_treffer / gesamt_bilder

print("-" * 50)
print("🎓 ERGEBNISSE DER ABSCHLUSSPRÜFUNG 🎓")
print(f"Test Loss (Fehlerwert): {test_loss_durchschnitt:.4f}")
print(f"Finale Test-Genauigkeit (Accuracy): {test_genauigkeit:.2f} %")
print(test_data.class_to_idx)
print("-" * 50)

Vorbereitung auf die Abschlussprüfung...
Es werden 580 unbekannte Bilder getestet.

Prüfung läuft... Bitte warten...
--------------------------------------------------
🎓 ERGEBNISSE DER ABSCHLUSSPRÜFUNG 🎓
Test Loss (Fehlerwert): 0.2035
Finale Test-Genauigkeit (Accuracy): 92.24 %
{'FAKE': 0, 'REAL': 1}
--------------------------------------------------


In [ ]:
import torch
import gc

# Löscht die Variablen, falls sie existieren
if 'model' in locals():
    del model
if 'optimizer' in locals():
    del optimizer

# Leert den RAM und den GPU-Speicher
gc.collect()
torch.cuda.empty_cache()

print("GPU-Speicher wurde bereinigt!")

GPU-Speicher wurde bereinigt!
